In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

LAG_RETURNS = 5
RSI_PERIOD = 14
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
SEQ_LEN = 30
LSTM_EPOCHS = 20

In [2]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    """Same as stack: LSTM features = return lags, volume lag, RSI, MACD. Target = next 21 returns."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        df["vix_lag_1"] = df["vix"].astype(float).shift(1)
    else:
        df["vix_lag_1"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    feature_cols_lstm = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + [
        "volume_lag_1", "rsi", "macd_line", "macd_signal"
    ]
    feature_cols_xgb = ["vix_lag_1", "month_sin", "month_cos", "fear_greed"]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return"] + feature_cols_lstm + feature_cols_xgb + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols_lstm, feature_cols_xgb, target_cols


def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    n = len(X)
    if n < seq_len + 1:
        return None, None
    X_seq, y_seq = [], []
    for i in range(seq_len, n):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

In [3]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    HAS_LSTM = True
except Exception:
    HAS_LSTM = False

def train_lstm(X_seq: np.ndarray, y_seq: np.ndarray, horizon: int, epochs=20):
    if not HAS_LSTM or X_seq is None or len(X_seq) < 10:
        return None
    n_feat = X_seq.shape[2]
    model = Sequential([
        LSTM(64, activation="tanh", input_shape=(X_seq.shape[1], n_feat)),
        Dense(horizon),
    ])
    model.compile(optimizer="adam", loss="mse")
    model.fit(X_seq, y_seq, epochs=epochs, verbose=0)
    return model

In [4]:
def train_global_lstm(stacked: pd.DataFrame, horizon: int):
    """Train one LSTM on pooled data (all assets, only rows before 60-day test window). Returns dict for predict_lstm_global."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols_lstm, _, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon or len(feat_df) < SEQ_LEN + horizon + 1:
            continue
        feat_dfs.append((feat_df, feature_cols_lstm, target_cols))
    if not feat_dfs:
        return None
    feature_cols_lstm = feat_dfs[0][1]
    target_cols = feat_dfs[0][2]
    X_list, y_list = [], []
    for feat_df, _, _ in feat_dfs:
        X_list.append(feat_df[feature_cols_lstm].values.astype(np.float32))
        y_list.append(feat_df[target_cols].values.astype(np.float32))
    X_all = np.vstack(X_list)
    y_all = np.vstack(y_list)
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_all)
    X_seq_list, y_seq_list = [], []
    for feat_df, _, _ in feat_dfs:
        X_sym = scaler.transform(feat_df[feature_cols_lstm].values.astype(np.float32))
        y_sym = feat_df[target_cols].values.astype(np.float32)
        X_seq, y_seq = build_sequences(X_sym, y_sym, SEQ_LEN)
        if X_seq is not None and len(X_seq) >= 10:
            X_seq_list.append(X_seq)
            y_seq_list.append(y_seq)
    if not X_seq_list:
        return None
    X_seq_all = np.concatenate(X_seq_list, axis=0)
    y_seq_all = np.concatenate(y_seq_list, axis=0)
    lstm_model = train_lstm(X_seq_all, y_seq_all, horizon, epochs=LSTM_EPOCHS)
    if lstm_model is None:
        return None
    return {"model": lstm_model, "scaler": scaler, "feature_cols_lstm": feature_cols_lstm}


def predict_lstm_global(context_df: pd.DataFrame, horizon: int, global_lstm: dict) -> list:
    """Predict 21 price steps using pre-trained global LSTM; no training."""
    if global_lstm is None:
        return []
    try:
        feat_df, feature_cols_lstm, _, _ = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < SEQ_LEN + 1:
        return []
    X = feat_df[feature_cols_lstm].values.astype(np.float32)
    X_s = global_lstm["scaler"].transform(X)
    last_idx = len(X_s) - 1
    if last_idx < SEQ_LEN:
        return []
    seq = X_s[last_idx - SEQ_LEN : last_idx]
    lstm_21 = global_lstm["model"].predict(seq.reshape(1, SEQ_LEN, -1), verbose=0).ravel()
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.cumprod(np.concatenate([[1.0], 1.0 + lstm_21]))[1:]
    return [float(p) for p in prices[:horizon]]

In [5]:
stacked = load_pool_data(with_vix=True, with_volume=True)
fear_greed_df = fetch_cnn_fear_greed_index(limit_days=1500)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = fear_greed_df["timestamp"]
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-09,AAPL,121.089996,129525800,24.030001,43.360000
1,2021-03-10,AAPL,119.980003,111943300,22.559999,45.560000
2,2021-03-11,AAPL,121.959999,103026500,21.910000,50.480000
3,2021-03-12,AAPL,121.029999,88105100,20.690001,53.720000
4,2021-03-15,AAPL,123.989998,92403800,20.030001,56.520000
5,2021-03-16,AAPL,125.570000,115227900,19.790001,54.800000
6,2021-03-17,AAPL,124.760002,111932600,19.230000,57.866667
7,2021-03-18,AAPL,120.529999,121229700,21.580000,52.333333
8,2021-03-19,AAPL,119.989998,185549500,20.950001,50.833333
9,2021-03-22,AAPL,123.389999,111912300,18.879999,50.700000


In [6]:
# Train once on pooled data (all assets, only rows before 60-day test window)
global_lstm = train_global_lstm(stacked, FORECAST_HORIZON)
print("Global LSTM trained on", len(build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)), "pooled train rows.")

c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Global LSTM trained on 11960 pooled train rows.


In [7]:
model_name = "lstm"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_lstm_global(context_df, FORECAST_HORIZON, global_lstm)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_lstm = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(
    columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"]
)
print(pred_lstm.groupby("symbol").size() if not pred_lstm.empty else "No predictions.")
pred_lstm.head()

symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-10,278.779999,277.872265,0,AAPL
1,2025-12-11,278.029999,277.945670,0,AAPL
2,2025-12-12,278.279999,278.515437,0,AAPL
3,2025-12-15,274.109985,279.325191,0,AAPL
4,2025-12-16,274.609985,277.836780,0,AAPL


In [8]:
metrics_rows = []
for sym in pred_lstm["symbol"].unique():
    sub = pred_lstm[pred_lstm["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_lstm)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_lstm_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_lstm_pool.parquet")

   model   symbol        MAE       RMSE    MAPE_%
0   lstm     AAPL  11.609806  13.935613  4.421445
1   lstm     MSFT  24.586888  27.934660  5.733179
2   lstm    GOOGL  13.245999  15.430681  4.144517
3   lstm     AMZN  13.262930  15.940824  6.075050
4   lstm      JPM  13.808767  15.515669  4.384934
5   lstm      JNJ  12.012337  13.516193  5.229763
6   lstm      WMT   5.678668   6.432132  4.635085
7   lstm      SPY   8.636814  10.403027  1.258589
8   lstm      XOM   7.898240   9.309527  5.646018
9   lstm     NVDA   5.186523   6.502189  2.836359
10  lstm  overall  11.592697  15.541788  4.436494
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_lstm_pool.parquet
